### 元素説明変数の頻出パターンマイニング



In [ ]:
import pandas as pd
import numpy as np
from pymatgen.core import Element

pd.set_option("display.max_rows", 1000)
pd.set_option('max_colwidth', 100)


原子物性特徴量データの読み込み。

In [ ]:
import json

with open("../data_calculated/atom_transaction.json","r") as f:
    g_transaction = json.load(f)


In [ ]:
g_transaction


In [ ]:
# それぞれのtransactionのitemの数を表示する．各transactionの個数がバラバラであることが分かる．
for name, value in g_transaction.items():
    print(name, len(value))



#### 頻出データマイニング

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from mlxtend.preprocessing import TransactionEncoder


In [ ]:
def get_transaction_df(transaction):
    te = TransactionEncoder()
    te.fit(transaction)
    te_ary = te.transform(transaction)
    df = pd.DataFrame(te_ary, columns=te.columns_)
    return df


g_df = get_transaction_df([v  for v in g_transaction.values()])


In [ ]:
g_df

supportが0.1（１０件以上）の頻出パターンマイニングを行う。

In [ ]:
g_df_freq_items = fpgrowth(g_df, min_support=0.1, use_colnames=True)
g_df_freq_items



#### ルールマイニング

- confidenceが0.8以上

となる要素を探す。


In [ ]:
g_df_rules = association_rules(
    g_df_freq_items, metric="confidence", min_threshold=0.8)

# itemの数を加える。
g_df_rules["antecedent_len"] = g_df_rules["antecedents"].apply(
    lambda x: len(x))
# supportが多いruleに並び直す。
g_df_rules = g_df_rules.sort_values(
    by="support", ascending=False).reset_index(drop=True)
g_df_rules.head(15)


図示すると物性値間の「規則」が分かります。
（見にくい場合は次を何度でも実行し直してください。図の配置が変わります。）

In [ ]:
def show_rules(df, figsize=(10, 10)):
    """show rules.
    
    Args:
        df (pd.DataFrame): data.
        figsize ((float,float), optional): figure size.
    """
    df = df.copy()

    df.antecedents = df.antecedents.apply(lambda x: next(iter(x)))
    df.consequents = df.consequents.apply(lambda x: next(iter(x)))

    import networkx as nx
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=figsize)
    GA = nx.from_pandas_edgelist(df,
                                 source='antecedents', target='consequents',
                                 create_using=nx.MultiDiGraph())
    nx.draw(GA, node_color="yellow", with_labels=True, ax=ax)
    fig.show()


show_rules(g_df_rules.iloc[:15, :])


- log10_electrical_resistivity<-6.90) $\Rightarrow$ (log10_thermal_conductivity>1.73) 

というルールが見える。
電気抵抗が小さい物質は熱伝導度も大きいということを
df_atomを検索して件数を確認するとともに図示する。

In [ ]:
# 元素特徴量の読み込み。
g_df_atom = pd.read_csv("data/atomicprop.csv", index_col=[0])
g_df_atom

In [ ]:
def query_rule(df_atom, ante, cons):
    """query rules.
    It simply show confidence. 

    Args:
        df_atom (pd.DataFrame): data.
        ante (str): antecedents.
        cons (str): consequents.
    """
    dfq_antecedents = df_atom.query(ante)
    dfq_consequents = df_atom.query("{} and {}".format(ante, cons))
    print("confidence {}/{}={}".format(dfq_consequents.shape[0],dfq_antecedents.shape[0],
          dfq_consequents.shape[0]/dfq_antecedents.shape[0]))


g_ante = "log10_electrical_resistivity<-6.90"
g_cons = "log10_thermal_conductivity>1.73"
query_rule(g_df_atom, g_ante, g_cons)


In [ ]:
from pymatgen.core.periodic_table import Element
import math
import matplotlib.pyplot as plt
%matplotlib inline


def plot_xy_symbol(df, x, y, ante=None, cons=None, filename=None):
    """plot symbols in 2D defined by x and y labels
       The points satisfying antecedents are colored in blue.
       The points satisfying antecedents and consequents are colored in red.

    Args:
        df (pd.DataFrame): data
        x (str): x label
        y (str): y label
        filename (str, optional): filename. Defaults to None.
    """
    fig, ax = plt.subplots()
    df = df.reset_index()
    dfq = df[[x, y, "index"]].dropna()
    for xval, yval, elm in zip(dfq[x].values,
                             dfq[y].values,
                             dfq["index"].values):
        if math.isnan(xval) or math.isnan(yval):
            pass
        else:
            ax.text(xval, yval, elm)
    ax.scatter(dfq[x].values,
               dfq[y].values, c="green")
    ax.set_xlabel(x)
    ax.set_ylabel(y)

    if ante is not None:
        print("antecedents={}".format(ante))
        df_ante = df.query(ante)
        ax.scatter(df_ante[x].values,
                   df_ante[y].values, c="blue")
    if ante is not None and cons is not None:
        print("consequents={}".format(cons))
        df_cons = df.query("{} and {}".format(ante, cons))
        ax.scatter(df_cons[x].values,
                   df_cons[y].values, c="r")
    if filename is not None:
        import os
        os.makedirs("image_executed", exist_ok=True)
        filepath = os.path.join("image_executed", filename)
        fig.savefig(filepath)
        print(filepath,"is made.")
    fig.show()


緑色が全体、青色＋赤色が前提部分（X）、赤色が結論部分（Y）を示す。

In [ ]:
# 順序相関関数を同時に出す。
from scipy.stats import spearmanr, pearsonr
def calc_correlation(df_atom,x,y):
    """
    nanを除いてspearman correlationを求める。
    
    Args:
        df_atom (pd.DataFrame): data.
        x (str): x column name.
        y (str): y column name.
    """
    df = df_atom[[x,y]].dropna()
    
    psr = pearsonr(df[x].values, df[y].values)
    spr  = spearmanr(df[x].values, df[y].values)
    print("Pearson R", psr)
    print( spr)

In [ ]:
plot_xy_symbol(g_df_atom, 'log10_electrical_resistivity', 'log10_thermal_conductivity',
               ante=g_ante, cons=g_cons, filename="atom_resistivity_vs_thermal_conductivity.png")
calc_correlation(g_df_atom, 'log10_electrical_resistivity', 'log10_thermal_conductivity',)

同様に、ルール
- (bulk_modulus>100.00) $\Rightarrow$	(molar_volume<12.29)
- (youngs_modulus>105.00) $\Rightarrow$	(molar_volume<12.29) 	
- (youngs_modulus>105.00) $\Leftrightarrow$	(bulk_modulus>100.00) 	

があるのでそれぞれの関係を図示する。


In [ ]:
g_ante = "bulk_modulus>100.00"
g_cons = "molar_volume<12.29"
plot_xy_symbol(g_df_atom, 'bulk_modulus', 'molar_volume',
               ante=g_ante, cons=g_cons)
calc_correlation(g_df_atom , 'bulk_modulus', 'molar_volume')

In [ ]:
g_ante = "youngs_modulus>105.00"
g_cons = "molar_volume<12.29"
plot_xy_symbol(g_df_atom, 'youngs_modulus', 'molar_volume',
               ante=g_ante, cons=g_cons)
calc_correlation(g_df_atom,'youngs_modulus', 'molar_volume')

In [ ]:
g_ante = "youngs_modulus>105.00"
g_cons = "bulk_modulus>100.00"
plot_xy_symbol(g_df_atom, 'youngs_modulus', 'bulk_modulus',
               ante=g_ante, cons=g_cons)
calc_correlation(g_df_atom,'youngs_modulus', 'bulk_modulus',)

In [ ]:
g_ante = "bulk_modulus>100.00"
g_cons = "youngs_modulus>105.00"
plot_xy_symbol(g_df_atom, 'bulk_modulus', 'youngs_modulus',
               ante=g_ante, cons=g_cons)
calc_correlation(g_df_atom,'bulk_modulus', 'youngs_modulus',)